# SUNSHIELD — Notebook 3: Anomaly Definition

**Place in Dr. Mai's workflow:** this notebook is the **"Anomaly definition using different σ"** box.
It turns each line into 0/1 — the **actual** line and the **predicted** line, each one **alone**:

```
Actual line    ─► anomaly definition (k = 1, 1.5, 2, 2.5, 3) ─► Actual 0/1     (one column per k)
Predicted line ─► anomaly definition (k = 1, 1.5, 2, 2.5, 3) ─► Predicted 0/1  (one column per k)
                                                                     └─► Notebook 4: F1 per k → best k
```

**The rule (Dr. Mai's sketch):** a reading is an anomaly (= 1) if it is far from the mean of **all the
readings before it** (from the first one, `Data[0 … t-1]`):

$$\text{anomaly}_t = 1 \quad \text{if} \quad x_t > \mu_{0..t-1} + k\,\sigma_{0..t-1} \;\; \text{or} \;\; x_t < \mu_{0..t-1} - k\,\sigma_{0..t-1}$$

| Step | What happens | Output |
|---|---|---|
| 1 | One function: series → 0/1 (1 = abnormal rise or drop, 0 = normal) | function + small test |
| 2 | Each point uses the mean and std of PREVIOUS readings only | — |
| 3 | Flag 1 if value > mean + k·std or value < mean − k·std | — |
| 4 | `MODE` setting: `"change"` (current − previous) or `"level"` (raw value) | setting in first cell |
| 5 | Apply to the ACTUAL line for k = 1, 1.5, 2, 2.5, 3 | `actual_anomaly` columns |
| 6 | Apply to the PREDICTED line separately (its own history), same k values | `predicted_anomaly` columns |
| 7 | Optional pruning using previous readings only | `PRUNING` setting |
| 8 | Save and zip outputs | `step3_outputs.zip` |

## Settings

In [1]:
import os, shutil
import numpy as np
import pandas as pd

MODE = 'level'              # 'level' : test the raw value (kW) — as in Dr. Mai's sketch
                            # 'change': test the change from the previous reading (current − previous)
K_VALUES = [1, 1.5, 2, 2.5, 3]
PRUNING = False             # True: a reading flagged as anomaly is left out of the mean/std of later readings
MIN_HISTORY = 2             # a std needs at least 2 previous readings; before that the reading is 0

OUT_DIR = 'step3_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
SPLITS = ['train', 'val', 'test']

## Upload Notebook 2 outputs

Run the cell below, click **Choose Files** and select `step2_outputs.zip`
(the file downloaded at the end of Notebook 2).

In [2]:
from google.colab import files
uploaded = files.upload()
shutil.unpack_archive(list(uploaded)[0], 'step2_outputs', 'zip')

predictions = {s: pd.read_csv(f'step2_outputs/prediction_{s}.csv', parse_dates=['DATE_TIME'])
                  .sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)
               for s in SPLITS}
for s in SPLITS:
    print(f'{s:5s}: {len(predictions[s]):,} readings, {predictions[s].groupby(["PLANT_ID", "SOURCE_KEY"]).ngroups} inverters')

train: 36,491 readings, 44 inverters
val  : 11,968 readings, 44 inverters
test : 10,758 readings, 44 inverters


## Steps 1–3 and 7 — The anomaly function

One function does everything. It walks through the series **in time order** and, for every reading:

1. computes the mean and std of the readings **before** it (never the reading itself, never the future),
2. flags 1 if the reading is above `mean + k·std` or below `mean − k·std`, otherwise 0,
3. adds the reading to the history for the next readings.

**`MODE = 'change'`** first replaces every value by its change from the previous reading
($x_t - x_{t-1}$), then applies the same rule — it catches sudden jumps and drops instead of high/low levels.

**`PRUNING = True`** (optional): a reading flagged as anomaly is **not** added to the history, so one big
spike does not inflate the std and hide the next anomalies. It still uses previous readings only.

The mean and std are kept as running sums (count, sum, sum of squares), so the function stays fast and
it is easy to see that only past readings are used.

In [3]:
def flag_anomalies(series, k, mode=MODE, pruning=PRUNING, min_history=MIN_HISTORY):
    '''series (in time order) -> numpy array of 0/1.
    1 = abnormal rise or drop: value outside mean ± k·std of the PREVIOUS readings only.'''
    x = np.asarray(series, dtype=float)
    if mode == 'change':
        x = np.diff(x, prepend=np.nan)          # change from the previous reading; the first has none
    elif mode != 'level':
        raise ValueError("MODE must be 'level' or 'change'")

    flags = np.zeros(len(x), dtype=int)
    n, total, total_sq = 0, 0.0, 0.0            # history so far: count, sum, sum of squares
    for t, value in enumerate(x):
        if np.isnan(value):
            continue
        if n >= min_history:
            mean = total / n
            std = np.sqrt(max((total_sq - n * mean ** 2) / (n - 1), 0.0))   # sample std, like pandas
            if value > mean + k * std or value < mean - k * std:
                flags[t] = 1
        if not (pruning and flags[t]):          # with pruning, anomalies stay out of the history
            n += 1
            total += value
            total_sq += value ** 2
    return flags

### Small test

The test series is a flat line with a big spike (index 6, a rise), a smaller spike (index 8) and a dip
(index 10, a drop), like Dr. Mai's sketch. It checks:
- the rise and the drop are found, the normal readings are not
- the function matches a plain pandas calculation of the mean/std of previous readings
- **no future readings are used:** changing the readings after index 6 does not change any flag up to index 6
- **pruning:** without it, the big spike inflates the std and hides the smaller spike at index 8;
  with `PRUNING = True` the big spike is left out of the history and index 8 is found too

In [4]:
test = np.array([10, 12, 9, 11, 10, 12, 40, 10, 25, 11, -15, 11], dtype=float)
flags = flag_anomalies(test, k=2, mode='level', pruning=False)
print('series                  :', test.astype(int).tolist())
print('flags                   :', flags.tolist())
assert flags[6] == 1 and flags[10] == 1, 'the rise and the drop must be flagged'
assert flags[[0, 1, 2, 3, 4, 5, 7, 8, 9, 11]].sum() == 0, 'normal readings must be 0'

# Same result as pandas: mean/std of the previous readings (shift(1) = only the past)
s = pd.Series(test)
mean_prev = s.expanding(MIN_HISTORY).mean().shift(1)
std_prev = s.expanding(MIN_HISTORY).std().shift(1)
pandas_flags = ((s > mean_prev + 2 * std_prev) | (s < mean_prev - 2 * std_prev)).astype(int).to_numpy()
assert (pandas_flags == flags).all()

# No future readings: change everything after index 6, the first 7 flags must stay the same
changed_future = test.copy()
changed_future[7:] = 1000
assert (flag_anomalies(changed_future, k=2, mode='level')[:7] == flags[:7]).all()

# Pruning: with the big spike kept out of the history, the std stays small and index 8 is found
pruned = flag_anomalies(test, k=2, mode='level', pruning=True)
print('flags with PRUNING=True :', pruned.tolist())
assert pruned[8] == 1

print("flags with MODE='change':", flag_anomalies(test, k=2, mode='change').tolist())
print('All tests passed.')

series                  : [10, 12, 9, 11, 10, 12, 40, 10, 25, 11, -15, 11]
flags                   : [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0]
flags with PRUNING=True : [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0]
flags with MODE='change': [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
All tests passed.


## Step 4 — `MODE` setting

`MODE` is set in the **first cell** (Settings). What each mode looks at:

| MODE | Value tested | Catches |
|---|---|---|
| `'level'` | the power itself (kW) | readings much higher or lower than usual |
| `'change'` | current − previous reading | sudden jumps and sudden drops |

In [5]:
print(f'MODE = {MODE!r} | PRUNING = {PRUNING} | k values = {K_VALUES}')

MODE = 'level' | PRUNING = False | k values = [1, 1.5, 2, 2.5, 3]


## Steps 5 and 6 — Apply to the ACTUAL line and to the PREDICTED line, each one alone

- Every **inverter** is its own series (as agreed in Notebook 1), in time order.
- Every **split** is handled on its own: the history starts at the first reading of the split
  (`Data[0]` in Dr. Mai's sketch). Night readings were removed in Notebook 1, so the history only
  contains daytime readings.
- The **actual** line uses the history of the actual line; the **predicted** line uses the history of
  the predicted line. The two lines never look at each other here — they only meet in Notebook 4.

Column names: `actual_anomaly_k1`, `actual_anomaly_k1.5`, … and `predicted_anomaly_k1`, …

In [6]:
LINES = {'actual': 'ACTUAL_AC_POWER_kW', 'predicted': 'PREDICTED_AC_POWER_kW'}

def add_anomaly_columns(df):
    out = df.copy()
    groups = out.groupby(['PLANT_ID', 'SOURCE_KEY'], sort=False).indices.values()
    for line, column in LINES.items():
        values = out[column].to_numpy()
        for k in K_VALUES:
            col = f'{line}_anomaly_k{k:g}'
            out[col] = 0
            for rows in groups:
                out.loc[rows, col] = flag_anomalies(values[rows], k)
    return out

anomalies = {s: add_anomaly_columns(predictions[s]) for s in SPLITS}
anomalies['train'].head()

,PLANT_ID,SOURCE_KEY,DATE_TIME,ACTUAL_AC_POWER_kW,PREDICTED_AC_POWER_kW,actual_anomaly_k1,actual_anomaly_k1.5,actual_anomaly_k2,actual_anomaly_k2.5,actual_anomaly_k3,predicted_anomaly_k1,predicted_anomaly_k1.5,predicted_anomaly_k2,predicted_anomaly_k2.5,predicted_anomaly_k3
0,4135001,1BY6WEcLGh8j5v7,2020-05-15 17:30:00,106.066667,128.681030,0,0,0,0,0,0,0,0,0,0
1,4135001,1BY6WEcLGh8j5v7,2020-05-15 17:45:00,47.012500,50.693085,0,0,0,0,0,0,0,0,0,0
2,4135001,1BY6WEcLGh8j5v7,2020-05-15 18:00:00,25.485714,19.852570,1,0,0,0,0,1,0,0,0,0
3,4135001,1BY6WEcLGh8j5v7,2020-05-15 18:15:00,7.542857,5.764679,1,0,0,0,0,1,0,0,0,0
4,4135001,1BY6WEcLGh8j5v7,2020-05-15 18:30:00,0.000000,-1.874908,1,0,0,0,0,0,0,0,0,0


In [7]:
# Share of readings flagged as anomaly, per split, line and k
rate = pd.DataFrame({
    (s, line): [100 * anomalies[s][f'{line}_anomaly_k{k:g}'].mean() for k in K_VALUES]
    for s in SPLITS for line in LINES}, index=[f'k = {k:g}' for k in K_VALUES])
rate.columns.names = ['split', 'line']
print('% of readings flagged as anomaly (1):')
rate.round(2)

% of readings flagged as anomaly (1):


split    train              val             test          
line    actual predicted actual predicted actual predicted
k = 1    42.30     43.27  36.82     37.68  41.05     39.06
k = 1.5  11.78     10.67  11.66     14.71  12.94     15.13
k = 2     3.28      2.02   4.90      4.78   5.86      3.89
k = 2.5   1.11      0.98   2.03      2.21   2.40      1.84
k = 3     0.49      0.60   0.80      0.37   1.29      0.82

## Step 8 — Save and zip outputs

| File | Content | Used by |
|---|---|---|
| `anomaly_train.csv` / `anomaly_val.csv` / `anomaly_test.csv` | actual + predicted kW, and the 0/1 columns for every k | **Notebook 4** |
| `anomaly_rate.csv` | % of readings flagged per split, line and k | report |
| `settings.txt` | `MODE`, `PRUNING`, k values used | report |

In [8]:
for s in SPLITS:
    anomalies[s].to_csv(os.path.join(OUT_DIR, f'anomaly_{s}.csv'), index=False)
rate.to_csv(os.path.join(OUT_DIR, 'anomaly_rate.csv'))
with open(os.path.join(OUT_DIR, 'settings.txt'), 'w') as f:
    f.write(f'MODE = {MODE}\nPRUNING = {PRUNING}\nK_VALUES = {K_VALUES}\nMIN_HISTORY = {MIN_HISTORY}\n')

shutil.make_archive('step3_outputs', 'zip', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f:22s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:6.2f} MB')

files.download('step3_outputs.zip')    # keep this file: Notebook 4 needs it

anomaly_rate.csv         0.00 MB
anomaly_test.csv         0.92 MB
anomaly_train.csv        3.12 MB
anomaly_val.csv          1.02 MB
settings.txt             0.00 MB
